# My Baseline Review Queue

I am confirming **Refresh / Content Opportunity Scoring** as my lane. I want to help an editor choose pages to inspect, starting with a rule that is easy to understand. This notebook checks two signals, freezes one rule, exports its queue, and reads the top ten critically.

I keep the March contract from Week 3: features from March 1–14, an assumed March 18 decision, and a March 18–31 decline outcome. The three-day reporting gap is an assumption. June is untouched. The shared `work/lane_utils.py` reproduces the frame from the pinned warehouse revision and caches it in an ignored parquet file.

## 1. Two signal checks and my rule's reasoning

I check signals on the **24 training clients only**, before deciding the rule. The later-window decline flag is used to evaluate each signal, never to calculate a score. The March split has already been explored in Week 3, so I call it a development comparison rather than a fresh final test.

**Signal 1 — search demand.** Volume sits behind FlyRank's quick-win logic. My question is whether higher earlier impressions also mean a higher rate of later decline. I use a page count `n` in every bucket. This is measured page demand, not keyword search-volume estimates.

**Signal 2 — low CTR within position bands.** CTR and position sit behind the CTR-review logic. I ask whether pages below the starter's 0.5% CTR threshold have more later decline within comparable position bands. These are broad bands, not query-level controls. The threshold is a working rule from the starter example, not a Google benchmark.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

needed = {"duckdb": "duckdb", "huggingface_hub": "huggingface_hub", "sklearn": "scikit-learn",
          "pyarrow": "pyarrow", "matplotlib": "matplotlib", "pandas": "pandas"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "work/lane_utils.py").is_file()), None)
if root is None:
    if importlib.util.find_spec("google.colab") is None:
        raise RuntimeError("Open this notebook inside your repository checkout.")
    root = Path("/content/FlyrankAI-Internship")
    if not root.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/vibhanshu-s/FlyrankAI-Internship.git", str(root)])
sys.path.insert(0, str(root / "work"))
import lane_utils as u
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pages = u.load_pages(root)
experiment = u.labelled_pages(pages)
train_index, test_index = u.grouped_split(experiment)
training = experiment.iloc[train_index].copy()
test = experiment.iloc[test_index].copy()
print(f"March review pool: {len(pages):,} pages; known outcomes: {len(experiment):,}; unknown: {pages.future_decline.isna().sum():,}.")
print(f"Fixed client split: {len(training):,} training pages / {len(test):,} test pages.")
print("Features stop March 14; assumed decision March 18; labels use March 18–31. June remains sealed.")

March review pool: 60,534 pages; known outcomes: 57,459; unknown: 3,075.
Fixed client split: 14,070 training pages / 43,389 test pages.
Features stop March 14; assumed decision March 18; labels use March 18–31. June remains sealed.


In [2]:
demand = training.copy()
demand["demand_bucket"] = pd.cut(demand.earlier_impressions,
    [99, 250, 500, 1000, 5000, np.inf], right=False,
    labels=["100–249", "250–499", "500–999", "1,000–4,999", "5,000+"])
demand_table = demand.groupby("demand_bucket", observed=False).agg(
    n=("future_decline", "size"), later_decline_rate=("future_decline", "mean"),
    median_earlier_impressions=("earlier_impressions", "median"))
display(demand_table.round(4))
print("Demand verdict: MIXED — the decline rates do not rise steadily with demand.")

capture = training.copy()
capture["position_bucket"] = pd.cut(capture.weighted_position, [0, 3, 10, 20, np.inf],
    labels=["1–3", "over 3–10", "over 10–20", "over 20"])
capture["ctr_bucket"] = np.where(capture.ctr_pct.lt(0.5), "below 0.5%", "0.5% or higher")
capture_table = capture.groupby(["position_bucket", "ctr_bucket"], observed=False).agg(
    n=("future_decline", "size"), later_decline_rate=("future_decline", "mean"),
    clicks=("earlier_clicks", "sum"), impressions=("earlier_impressions", "sum"))
capture_table["weighted_ctr_pct"] = 100 * capture_table.clicks / capture_table.impressions
display(capture_table[["n", "later_decline_rate", "weighted_ctr_pct"]].round(4))
print("CTR-with-position verdict: CONFIRMED — lower CTR has higher decline rates in each training band.")

,n,later_decline_rate,median_earlier_impressions
demand_bucket,,,
100–249,4012,0.2884,161.0
250–499,2961,0.3793,347.0
500–999,3194,0.2752,782.0
"1,000–4,999",3216,0.3315,1795.5
"5,000+",687,0.3377,8127.0


Demand verdict: MIXED — the decline rates do not rise steadily with demand.


n  later_decline_rate  weighted_ctr_pct
position_bucket ctr_bucket                                                
1–3             0.5% or higher   342              0.2953            0.9801
                below 0.5%       572              0.4056            0.2086
over 3–10       0.5% or higher  2083              0.2444            0.9670
                below 0.5%      5693              0.3005            0.2142
over 10–20      0.5% or higher   655              0.2458            0.8588
                below 0.5%      2093              0.3349            0.1746
over 20         0.5% or higher   276              0.2899            0.7406
                below 0.5%      2356              0.4083            0.1334

CTR-with-position verdict: CONFIRMED — lower CTR has higher decline rates in each training band.


**My verdicts:** Demand is **MIXED** as a decline-risk signal: the training rates move up and down across the buckets. I still use demand to represent how much existing visibility is at stake, not as proof a page will decline. CTR-with-position is **CONFIRMED** for this training comparison: below-0.5% pages have higher decline rates in every displayed band. Client mix and query intent could still explain part of that association.

**My one rule:** Start with `log(1 + earlier impressions)`. Double that score when earlier CTR is below 0.5% and the impression-weighted position is between 1 and 20. Recommend a review of search performance, with the single reason code `visible_page_review`.

The multiplier is a plain hand-written priority choice, not a fitted coefficient. All pages already meet the 100-impression and 14-day coverage floor. The log limits how much a large page dominates. I freeze the formula here; I will not move its thresholds after seeing the model results.

## 2. Build the ranked queue

I rank within each client, using a pseudonymous page ID only to break score ties. The action is **review search performance**: inspect the search snippet, search intent, content accuracy, and indexing context before deciding whether to edit. The score is not a probability of decline or a promised traffic gain.

The CSV includes only earlier measurements, score, action, reason, and pseudonymous keys. Later measurements and labels stay in the evaluation frame. Missing outcomes do not remove a page from this review queue.

In [3]:
rule_inputs = pages[["log_impressions", "ctr_pct", "weighted_position"]].copy()
scores = u.rule_score(rule_inputs)
queue = u.rank_frame(pages[u.KEYS + u.FEATURES + ["earlier_impressions"]], scores)
queue["reason_code"] = u.BASELINE["reason_code"]
queue["action"] = u.BASELINE["action"]
output = root / "work/outputs/baseline_action_score.csv"
queue.to_csv(output, index=False)
baseline_metrics = u.evaluate(test, u.rule_score(test))
receipt = {
    "contract": u.CONTRACT, "data_fingerprint": u.fingerprint(pages),
    "split": u.split_fingerprint(experiment, train_index, test_index),
    "baseline": u.BASELINE, "signal_verdicts": {"demand": "MIXED", "ctr_with_position": "CONFIRMED"},
    "training_bucket_counts": demand_table.n.astype(int).tolist(),
    "metrics": baseline_metrics, "export_rows": len(queue),
}
u.write_receipt(root, "w04_baseline_metrics.json", receipt)
display(u.summary_table({"Frozen rule": baseline_metrics}))
print(f"Wrote {len(queue):,} earlier-only queue rows to work/outputs/baseline_action_score.csv.")
print("Macro base rate is shown for the same full-queue clients; shorter queues are reported separately.")
print(f"Short queues: {baseline_metrics['short_queue_clients']}; mean decline share: {baseline_metrics['short_queue_mean']:.4f}.")

,macro_precision_at_20,macro_base_rate_same_clients,roc_auc,average_precision,full_queue_clients,short_queue_clients,pages
Frozen rule,0.3714,0.3981,0.516,0.3811,7.0,1.0,43389.0


Wrote 60,534 earlier-only queue rows to work/outputs/baseline_action_score.csv.
Macro base rate is shown for the same full-queue clients; shorter queues are reported separately.
Short queues: 1; mean decline share: 0.3333.


**Starting result:** The frozen rule reaches **37.14% mean precision@20**, compared with a **39.81% mean decline rate** in the same full-queue clients' eligible pools. It does not concentrate decline better than that reference in this development check. I keep the rule unchanged so Week 5 has a fair baseline to beat. This also shows why a training signal verdict is not enough to claim a useful review system.

## 3. Top-10 review

These are the ten highest scores across the queue for a close reading of the rule. They are not a global allocation of editor time: actual review capacity stays within each client. I can inspect measurements here, not actual page text, so these are provisional reviews rather than ten verified editing recommendations.

Each line below gives the action, measured reason, confidence note, and a different way the recommendation could be wrong.

In [4]:
top10 = queue.sort_values(["score", "content_hash_id"], ascending=[False, True]).head(10)
failure_checks = [
    "the query mix may be mainly informational and answered directly on the results page",
    "brand and non-brand queries may be mixed, hiding a healthy snippet",
    "the average position may hide very different mobile and desktop results",
    "a short-lived seasonal surge may explain the volume",
    "rich results or other result-page features may reduce clicks without a content problem",
    "the page may already answer its intended audience well and need no rewrite",
    "the URL or indexing setup may be the issue instead of the content",
    "the search demand may have shifted away from the original topic",
    "the large client may dominate a global list despite a smaller local review budget",
    "the aggregate position may hide country or query changes that a snippet edit will not fix",
]
lines = []
for number, ((_, row), failure) in enumerate(zip(top10.iterrows(), failure_checks), 1):
    boost = row.ctr_pct < 0.5 and 1 <= row.weighted_position <= 20
    why = f"{row.earlier_impressions:,.0f} earlier impressions, {row.ctr_pct:.3f}% CTR, position {row.weighted_position:.1f}"
    why += "; low-CTR boost applies" if boost else "; demand score only"
    lines.append(f"{number}. **{row.content_hash_id}** — Review search performance; {why}. "
                 f"Reason: `visible_page_review`. Confidence: provisional. Wrong if {failure}.")
assert len(lines) == 10
display(Markdown((chr(10) * 2).join(lines)))

1. **content_e8a52cf3d5988c07** — Review search performance; 138,793 earlier impressions, 0.245% CTR, position 16.2; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if the query mix may be mainly informational and answered directly on the results page.

2. **content_7172a7fad43f0998** — Review search performance; 102,917 earlier impressions, 0.444% CTR, position 3.0; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if brand and non-brand queries may be mixed, hiding a healthy snippet.

3. **content_b99ea6861864dea5** — Review search performance; 86,624 earlier impressions, 0.197% CTR, position 4.2; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if the average position may hide very different mobile and desktop results.

4. **content_7c6373141eae744a** — Review search performance; 84,160 earlier impressions, 0.056% CTR, position 6.0; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if a short-lived seasonal surge may explain the volume.

5. **content_acbcc847f8996314** — Review search performance; 77,413 earlier impressions, 0.155% CTR, position 3.5; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if rich results or other result-page features may reduce clicks without a content problem.

6. **content_471d9cabce329a66** — Review search performance; 75,458 earlier impressions, 0.256% CTR, position 4.4; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if the page may already answer its intended audience well and need no rewrite.

7. **content_fd2117c2c6790e4b** — Review search performance; 73,817 earlier impressions, 0.263% CTR, position 3.7; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if the URL or indexing setup may be the issue instead of the content.

8. **content_5e1c049f62e33b11** — Review search performance; 70,353 earlier impressions, 0.144% CTR, position 17.7; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if the search demand may have shifted away from the original topic.

9. **content_82e35c4845e6c391** — Review search performance; 67,922 earlier impressions, 0.040% CTR, position 17.8; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if the large client may dominate a global list despite a smaller local review budget.

10. **content_34a70fea29d15f24** — Review search performance; 67,555 earlier impressions, 0.027% CTR, position 2.8; low-CTR boost applies. Reason: `visible_page_review`. Confidence: provisional. Wrong if the aggregate position may hide country or query changes that a snippet edit will not fix.

## 4. Weak picks and leakage check

Low CTR is a useful reason to investigate, but it cannot identify what is wrong with a page. The top ten can concentrate in one client, and a high-demand page may have a healthy reason for low click capture. The fixed 0.5% cutoff also ignores query intent and country/device differences. Those are reasons to keep human review in the action label.

The weak picks below are selected pages whose later impressions did **not** meet the decline target. This is a retrospective check of errors, not information used to rank the queue. Non-decline does not prove review was worthless; it shows why this proxy and the action are different things.

In [5]:
ranked_test = u.rank_frame(test, u.rule_score(test))
weak = ranked_test.loc[ranked_test.client_rank.le(20) & ranked_test.future_decline.eq(0)]
display(weak[u.KEYS + ["client_rank", "score", "ctr_pct", "weighted_position", "future_decline"]].head(3))
print(f"Selected test pages without observed decline: {len(weak)}.")
assert np.array_equal(scores, u.rule_score(rule_inputs))
assert set(rule_inputs.columns).isdisjoint({"future_decline", "later_impressions", "later_days", "leaked_future_decline"})
assert queue.reason_code.nunique() == queue.action.nunique() == 1
assert set(queue.columns).isdisjoint({"future_decline", "later_impressions", "later_days"})
print("Checked: the score and exported queue contain no later-window or label-derived inputs.")

,client_hash_id,content_hash_id,client_rank,score,ctr_pct,weighted_position,future_decline
6687,client_2094c6eb080311d5,content_b0075b5c59c1de56,1,14.988860,0.166945,10.579299,0
6714,client_2094c6eb080311d5,content_d3b2fa9ea6547672,3,13.843316,0.197433,10.199408,0
6682,client_2094c6eb080311d5,content_aad79dde38dce898,5,13.725516,0.104712,7.663874,0


Selected test pages without observed decline: 90.
Checked: the score and exported queue contain no later-window or label-derived inputs.


## 5. Self-check

- [x] I confirmed my lane and checked two signals using training clients only.
- [x] Both bucket tables show `n`; at least one signal comes from real FlyRank flag logic.
- [x] I gave each signal one verdict and explained the negative or limited evidence.
- [x] I froze one rule with a score, one reason code, and one review action.
- [x] The notebook writes the ignored queue CSV and an aggregate metrics receipt.
- [x] I reviewed ten actual rows with a reason and a possible failure for each.
- [x] The rule uses earlier measurements only; the notebook ran top to bottom.
- [x] The executed notebook and metrics JSON are included in the task commit; no dataset is committed.